# CodeAlpha Machine Learning Internship — Task 3
# Handwritten Character Recognition (CNN)

**Objective:** Identify handwritten digits using a Convolutional Neural Network (CNN).

**Dataset:** MNIST (60,000 train / 10,000 test images, 28x28 grayscale digits 0-9)

> ⚠️ **Run this notebook in [Google Colab](https://colab.research.google.com/)** for a zero-setup environment with TensorFlow and internet access pre-installed (needed to download MNIST). Colab also gives free GPU access which speeds up training significantly.
>
> To extend this to full alphabet characters, swap in the **EMNIST** dataset (`byclass` or `letters` split) — the CNN architecture below works unchanged.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
keras.utils.set_random_seed(42)

## 2. Load & Preprocess MNIST Data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1] and add channel dimension (grayscale = 1 channel)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test:  {x_test.shape}, y_test:  {y_test.shape}")

In [ ]:
# Visualize a few sample digits
plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Build the CNN Architecture

Two convolutional blocks (Conv2D + BatchNorm + MaxPooling) followed by a dense classifier head with dropout for regularization.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=2),

    layers.Conv2D(128, kernel_size=3, activation="relu", padding="same"),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train the Model

In [ ]:
callbacks = [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    callbacks=callbacks,
    verbose=2,
)

## 5. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
y_pred = np.argmax(model.predict(x_test), axis=1)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - CNN on MNIST")
plt.show()

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Model Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Model Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Sample Predictions

In [ ]:
n = 15
preds = np.argmax(model.predict(x_test[:n]), axis=1)
plt.figure(figsize=(15, 4))
for i in range(n):
    plt.subplot(2, 8, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    color = "green" if preds[i] == y_test[i] else "red"
    plt.title(f"P:{preds[i]} / T:{y_test[i]}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Save the Model

In [ ]:
model.save("models/cnn_mnist_model.keras")
print("Model saved.")

## 9. Conclusion

- A CNN with 3 convolutional blocks achieves **~99% test accuracy** on MNIST — a well-established benchmark result for this architecture style.
- Batch normalization and dropout help the model generalize and avoid overfitting.
- **Next steps to extend this project:**
  - Swap in **EMNIST** for full alphabet character recognition (not just digits)
  - Use a **CRNN** (CNN + RNN) architecture to recognize full words/sentences instead of single characters
  - Deploy the model with a simple Flask/Streamlit app where users can draw a digit and get a live prediction